In [1]:
import pandas as pd
import numpy as np
import itertools
import datetime
import pandas_gbq
import matplotlib.pyplot as plt
from datetime import *
from datetime import datetime, timedelta, date
from pathlib import Path
from PIL import Image
# %load_ext google.colab.data_table
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
project_id = "perceptive-ivy-290216"

# Standard plotly imports
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 100)

In [2]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [3]:
query2=f"""
SELECT
*
FROM `perceptive-ivy-290216.f1_api.sprint_lap_time`  A
# WHERE A.Year=2024
# AND A.GP="Monaco Grand Prix"
# AND A.DRIVER='HAM'
ORDER BY LapNumber, LapStartTime
"""
track3=pandas_gbq.read_gbq(query2,project_id,dialect='standard')

Downloading: 100%|██████████|


In [4]:
track2=track3[(track3["GP"]=='Qatar Grand Prix')&(track3["Year"]==2024)]
track2.head()
year=track2['Year'].iloc[0]
gp=track2['GP'].iloc[0]

In [5]:
track2['LapTime']= pd.to_timedelta(track2["LapTime"])

/var/folders/x_/b65sxrpx6737wtqnt0ctv71w0000gn/T/ipykernel_21106/4173085553.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  track2['LapTime']= pd.to_timedelta(track2["LapTime"])


In [6]:
# track2['LapTime'].dt.total_seconds().min()*1.07
track2=track2[track2['LapNumber']!=1.0]

In [7]:
# quicklaps=track2[track2["LapTime"]<track2['LapTime'].min()*1.1]
quicklaps=track2
quicklaps.head()

,Time,Driver,DriverNumber,LapTime,LapNumber,Stint,PitOutTime,PitInTime,Sector1Time,Sector2Time,Sector3Time,Sector1SessionTime,Sector2SessionTime,Sector3SessionTime,SpeedI1,SpeedI2,SpeedFL,SpeedST,IsPersonalBest,Compound,TyreLife,FreshTyre,Team,LapStartTime,LapStartDate,TrackStatus,Position,Deleted,DeletedReason,FastF1Generated,IsAccurate,Year,GP
406,0 days 00:47:07.164000,NOR,4,0 days 00:01:25.961000,2.0,1.0,NaT,NaT,0 days 00:00:31.925000,0 days 00:00:29.161000,0 days 00:00:24.875000,0 days 00:46:13.128000,0 days 00:46:42.289000,0 days 00:47:07.164000,243.0,284.0,273.0,289.0,True,MEDIUM,7.0,False,McLaren,0 days 00:45:41.203000,2024-11-30 14:04:55.542,1,1.0,False,,False,True,2024,Qatar Grand Prix
409,0 days 00:47:08.279000,PIA,81,0 days 00:01:25.982000,2.0,1.0,NaT,NaT,0 days 00:00:32.133000,0 days 00:00:29.102000,0 days 00:00:24.747000,0 days 00:46:14.441000,0 days 00:46:43.543000,0 days 00:47:08.290000,239.0,284.0,278.0,294.0,True,MEDIUM,7.0,False,McLaren,0 days 00:45:42.297000,2024-11-30 14:04:56.636,1,2.0,False,,False,True,2024,Qatar Grand Prix
411,0 days 00:47:08.962000,RUS,63,0 days 00:01:26.166000,2.0,1.0,NaT,NaT,0 days 00:00:32.122000,0 days 00:00:29.328000,0 days 00:00:24.716000,0 days 00:46:14.882000,0 days 00:46:44.210000,0 days 00:47:08.926000,243.0,288.0,281.0,295.0,True,MEDIUM,6.0,False,Mercedes,0 days 00:45:42.796000,2024-11-30 14:04:57.135,1,3.0,False,,False,True,2024,Qatar Grand Prix
414,0 days 00:47:10.116000,SAI,55,0 days 00:01:26.295000,2.0,1.0,NaT,NaT,0 days 00:00:32.062000,0 days 00:00:29.359000,0 days 00:00:24.874000,0 days 00:46:15.888000,0 days 00:46:45.247000,0 days 00:47:10.121000,242.0,286.0,278.0,296.0,True,MEDIUM,3.0,False,Ferrari,0 days 00:45:43.821000,2024-11-30 14:04:58.160,1,4.0,False,,False,True,2024,Qatar Grand Prix
416,0 days 00:47:10.929000,HAM,44,0 days 00:01:26.545000,2.0,1.0,NaT,NaT,0 days 00:00:32.450000,0 days 00:00:29.264000,0 days 00:00:24.831000,0 days 00:46:16.825000,0 days 00:46:46.089000,0 days 00:47:10.920000,239.0,287.0,280.0,297.0,True,MEDIUM,8.0,False,Mercedes,0 days 00:45:44.384000,2024-11-30 14:04:58.723,1,5.0,False,,False,True,2024,Qatar Grand Prix


In [38]:
#Remove Pitstops and Track Status other than Clear to remove slow laps
quicklaps=quicklaps[((quicklaps["PitOutTime"]=='NaT')&(quicklaps["PitInTime"]=='NaT')&(~quicklaps["TrackStatus"].isin(['4','41','5','6','7','124'])))]

In [39]:
transformed_laps_driver = quicklaps.copy()
transformed_laps_driver.loc[:, "LapTime (s)"] = quicklaps["LapTime"].dt.total_seconds()

# order the team from the fastest (lowest median lap time) tp slower
team_order = (
    transformed_laps_driver[["Driver", "LapTime (s)"]].groupby("Driver").median()["LapTime (s)"].sort_values().index
)
print(team_order)

Index(['PIA', 'LEC', 'RUS', 'SAI', 'VER', 'NOR', 'HAM', 'GAS', 'HUL', 'MAG',
       'ALO', 'BOT', 'STR', 'COL', 'PER', 'OCO', 'TSU', 'LAW', 'ALB', 'ZHO'],
      dtype='object', name='Driver')


In [ ]:
transformed_laps_driver['Median (s)'] = transformed_laps_driver.groupby(['Driver','Compound'])["LapTime (s)"].transform('median')
transformed_laps_driver['Median LapTime'] = transformed_laps_driver.groupby(['Driver','Compound'])["LapTime"].transform('median')

transformed_laps_driver['Fastest (s)'] = transformed_laps_driver.groupby(['Driver','Compound'])["LapTime (s)"].transform('min')
transformed_laps_driver['Fastest LapTime'] = transformed_laps_driver.groupby(['Driver','Compound'])["LapTime"].transform('min')

transformed_laps_driver['Average (s)'] = transformed_laps_driver.groupby(['Driver','Compound'])["LapTime (s)"].transform(np.mean)
transformed_laps_driver['Average LapTime'] = transformed_laps_driver.groupby(['Driver','Compound'])["LapTime"].transform(np.mean)

transformed_laps_driver=transformed_laps_driver.fillna(0)

transformed_laps_driver.tail()

/var/folders/x_/b65sxrpx6737wtqnt0ctv71w0000gn/T/ipykernel_21106/3103475934.py:7: FutureWarning: The provided callable <function mean at 0x120af3920> is currently using SeriesGroupBy.mean. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "mean" instead.
  transformed_laps_driver['Average (s)'] = transformed_laps_driver.groupby(['Driver','Compound'])["LapTime (s)"].transform(np.mean)
/var/folders/x_/b65sxrpx6737wtqnt0ctv71w0000gn/T/ipykernel_21106/3103475934.py:8: FutureWarning: The provided callable <function mean at 0x120af3920> is currently using SeriesGroupBy.mean. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "mean" instead.
  transformed_laps_driver['Average LapTime'] = transformed_laps_driver.groupby(['Driver','Compound'])["LapTime"].transform(np.mean)


,Time,Driver,DriverNumber,LapTime,LapNumber,Stint,PitOutTime,PitInTime,Sector1Time,Sector2Time,Sector3Time,Sector1SessionTime,Sector2SessionTime,Sector3SessionTime,SpeedI1,SpeedI2,SpeedFL,SpeedST,IsPersonalBest,Compound,TyreLife,FreshTyre,Team,LapStartTime,LapStartDate,TrackStatus,Position,Deleted,DeletedReason,FastF1Generated,IsAccurate,Year,GP,LapTime (s),Median (s),Median LapTime,Fastest (s),Fastest LapTime,Average (s),Average LapTime
6205,0 days 01:11:50.884000,LAW,30,0 days 00:01:26.458000,19.0,1.0,NaT,NaT,0 days 00:00:32.090000,0 days 00:00:29.361000,0 days 00:00:25.007000,0 days 01:10:56.514000,0 days 01:11:25.875000,0 days 01:11:50.882000,237.0,285.0,276.0,0.0,False,MEDIUM,26.0,False,RB,0 days 01:10:24.426000,2024-11-30 14:29:38.765,1,16.0,False,,False,True,2024,Qatar Grand Prix,86.458,86.4110,0 days 00:01:26.411000,85.762,0 days 00:01:25.762000,86.672833,0 days 00:01:26.672833333
6206,0 days 01:11:51.630000,TSU,22,0 days 00:01:26.115000,19.0,1.0,NaT,NaT,0 days 00:00:31.950000,0 days 00:00:29.169000,0 days 00:00:24.996000,0 days 01:10:57.460000,0 days 01:11:26.629000,0 days 01:11:51.625000,0.0,288.0,279.0,303.0,False,MEDIUM,26.0,False,RB,0 days 01:10:25.515000,2024-11-30 14:29:39.854,1,17.0,False,,False,True,2024,Qatar Grand Prix,86.115,86.3895,0 days 00:01:26.389500,85.838,0 days 00:01:25.838000,86.663333,0 days 00:01:26.663333333
6207,0 days 01:11:52.167000,COL,43,0 days 00:01:25.959000,19.0,1.0,NaT,NaT,0 days 00:00:31.871000,0 days 00:00:29.355000,0 days 00:00:24.733000,0 days 01:10:58.095000,0 days 01:11:27.450000,0 days 01:11:52.183000,0.0,286.0,282.0,324.0,False,MEDIUM,15.0,True,Williams,0 days 01:10:26.208000,2024-11-30 14:29:40.547,1,18.0,False,,False,True,2024,Qatar Grand Prix,85.959,86.1530,0 days 00:01:26.153000,85.599,0 days 00:01:25.599000,86.286400,0 days 00:01:26.286400
6208,0 days 01:12:27.964000,ZHO,24,0 days 00:01:25.051000,19.0,2.0,NaT,NaT,0 days 00:00:31.607000,0 days 00:00:28.761000,0 days 00:00:24.683000,0 days 01:11:34.487000,0 days 01:12:03.248000,0 days 01:12:27.931000,245.0,286.0,276.0,286.0,True,MEDIUM,17.0,False,Kick Sauber,0 days 01:11:02.913000,2024-11-30 14:30:17.252,1,19.0,False,,False,True,2024,Qatar Grand Prix,85.051,85.7055,0 days 00:01:25.705500,85.051,0 days 00:01:25.051000,85.714875,0 days 00:01:25.714875
6209,0 days 01:12:30.899000,PER,11,0 days 00:01:24.929000,19.0,2.0,NaT,NaT,0 days 00:00:31.450000,0 days 00:00:28.829000,0 days 00:00:24.650000,0 days 01:11:37.404000,0 days 01:12:06.233000,0 days 01:12:30.883000,241.0,286.0,280.0,296.0,False,MEDIUM,26.0,False,Red Bull Racing,0 days 01:11:05.970000,2024-11-30 14:30:20.309,1,20.0,False,,False,True,2024,Qatar Grand Prix,84.929,86.2170,0 days 00:01:26.217000,84.892,0 days 00:01:24.892000,86.382385,0 days 00:01:26.382384615


In [49]:
transformed_driver=transformed_laps_driver.groupby(["Year","GP","Driver","Team","Compound"])[['LapTime (s)',
       'Median (s)', 'Median LapTime', 'Fastest (s)', 'Fastest LapTime',
       'Average (s)', 'Average LapTime']].min()

In [50]:
transformed_driver

LapTime (s)  \
Year GP               Driver Team            Compound                
2024 Qatar Grand Prix ALB    Williams        MEDIUM         85.443   
                      ALO    Aston Martin    MEDIUM         84.281   
                      BOT    Kick Sauber     MEDIUM         85.447   
                                             nan            87.122   
                      COL    Williams        MEDIUM         85.599   
                                             nan            86.601   
                      GAS    Alpine          MEDIUM         84.930   
                      HAM    Mercedes        MEDIUM         84.337   
                      HUL    Haas F1 Team    MEDIUM         84.284   
                      LAW    RB              MEDIUM         85.762   
                      LEC    Ferrari         MEDIUM         83.923   
                      MAG    Haas F1 Team    MEDIUM         84.568   
                      NOR    McLaren         MEDIUM         84.329   
                      OCO    Alpine          MEDIUM         85.598   
                      PER    Red Bull Racing MEDIUM         84.892   
                                             nan            86.185   
                      PIA    McLaren         MEDIUM         84.454   
                      RUS    Mercedes        MEDIUM         84.380   
                      SAI    Ferrari         MEDIUM         84.405   
                      STR    Aston Martin    MEDIUM         85.369   
                      TSU    RB              MEDIUM         85.838   
                      VER    Red Bull Racing MEDIUM         84.577   
                      ZHO    Kick Sauber     MEDIUM         85.051   
                                             SOFT           87.435   

                                                       Median (s)  \
Year GP               Driver Team            Compound               
2024 Qatar Grand Prix ALB    Williams        MEDIUM       86.4225   
                      ALO    Aston Martin    MEDIUM       85.7165   
                      BOT    Kick Sauber     MEDIUM       85.7370   
                                             nan          87.5850   
                      COL    Williams        MEDIUM       86.1530   
                                             nan          86.6900   
                      GAS    Alpine          MEDIUM       85.4675   
                      HAM    Mercedes        MEDIUM       85.4480   
                      HUL    Haas F1 Team    MEDIUM       85.4770   
                      LAW    RB              MEDIUM       86.4110   
                      LEC    Ferrari         MEDIUM       85.2435   
                      MAG    Haas F1 Team    MEDIUM       85.5360   
                      NOR    McLaren         MEDIUM       85.4030   
                      OCO    Alpine          MEDIUM       86.3690   
                      PER    Red Bull Racing MEDIUM       86.2170   
                                             nan          86.2730   
                      PIA    McLaren         MEDIUM       85.0740   
                      RUS    Mercedes        MEDIUM       85.2800   
                      SAI    Ferrari         MEDIUM       85.2940   
                      STR    Aston Martin    MEDIUM       85.9085   
                      TSU    RB              MEDIUM       86.3895   
                      VER    Red Bull Racing MEDIUM       85.3800   
                      ZHO    Kick Sauber     MEDIUM       85.7055   
                                             SOFT         87.8960   

                                                              Median LapTime  \
Year GP               Driver Team            Compound                          
2024 Qatar Grand Prix ALB    Williams        MEDIUM   0 days 00:01:26.422500   
                      ALO    Aston Martin    MEDIUM   0 days 00:01:25.716500   
                      BOT    Kick Sauber     MEDIUM   0 days 00:01:25.737000   
                                       